# Analysis - Priority Banding Simulations

In [66]:
import pandas as pd
import pickle
import json
import os
import datetime as dt
from matplotlib import pyplot as plt
import re
import numpy as np

In [59]:
banding_out = "D:/lco/custom_scheduler/output_files/banding"
banding_in = "D:/lco/custom_scheduler/input_files/banding"
baseline_out = "D:/lco/custom_scheduler/output_files/baseline"
print(os.path.isdir(banding_out))
print(os.path.isdir(banding_in))
print(os.path.isdir(baseline_out))

True
True
True


In [60]:
filemap = {}
for dirname, folders, filenames in os.walk(banding_in):
    for filename in filenames:
        output_filepath = os.path.join(banding_out, filename)
        forename, ext = os.path.splitext(filename)
        perfect_filepath = os.path.join(banding_out, forename + "_perfect" + ext)
        print(baseline_out)
        baseline_filepath = os.path.join(baseline_out, "_".join(forename.split("_")[:-1]).replace("pb_", "baseline_")+ext)
        # baseline_filepath = os.path.join(baseline_out, filename.replace("pb_", "baseline_"))
        print(baseline_filepath)
        print(filename)
        print(os.path.isfile(output_filepath))
        print(os.path.isfile(perfect_filepath))
        print(os.path.isfile(baseline_filepath))
        filemap[filename] = {
            "input": os.path.join(dirname, filename),
            "output": output_filepath,
            "perfect": perfect_filepath,
            "baseline": baseline_filepath
        }

D:/lco/custom_scheduler/output_files/baseline
D:/lco/custom_scheduler/output_files/baseline\baseline_2020-08.pkl
pb_2020-08_100-30-10.pkl
True
True
True
D:/lco/custom_scheduler/output_files/baseline
D:/lco/custom_scheduler/output_files/baseline\baseline_2020-08.pkl
pb_2020-08_100-55-10.pkl
True
True
True
D:/lco/custom_scheduler/output_files/baseline
D:/lco/custom_scheduler/output_files/baseline\baseline_2020-08.pkl
pb_2020-08_30-20-10.pkl
True
True
True
D:/lco/custom_scheduler/output_files/baseline
D:/lco/custom_scheduler/output_files/baseline\baseline_2021-02.pkl
pb_2021-02_100-30-10.pkl
True
True
True
D:/lco/custom_scheduler/output_files/baseline
D:/lco/custom_scheduler/output_files/baseline\baseline_2021-02.pkl
pb_2021-02_100-55-10.pkl
True
True
True
D:/lco/custom_scheduler/output_files/baseline
D:/lco/custom_scheduler/output_files/baseline\baseline_2021-02.pkl
pb_2021-02_30-20-10.pkl
True
True
True
D:/lco/custom_scheduler/output_files/baseline
D:/lco/custom_scheduler/output_files/b

In [86]:
calib_proposals = [
    "OGG_calib",
    "MuSCAT Commissioning",
    "auto_focus",
    "LCOEngineering",
    "COJ_calib",
    "FLOYDS standards",
    "Photometric standards",
    "standard"
]

In [93]:
for filename in filemap:
    i = pickle.load(open(filemap[filename]["input"], "rb")) #input
    o = pickle.load(open(filemap[filename]["output"], "rb")) #output
    p = pickle.load(open(filemap[filename]["perfect"], "rb")) #perfect
    b = pickle.load(open(filemap[filename]["baseline"], "rb")) #baseline

    proposals = i["proposals"]
    for calib_name in calib_proposals:
        if calib_name in proposals:
            proposals[calib_name] = 0

    data = i["all_requests"][["id", "proposal_id"]]
    data["priority"] = data["proposal_id"].apply(lambda x: proposals[x])
    data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
    data["baseline"] = data["id"].isin(b["final_completed_requests"].keys())
    data["perfect"] = data["id"].isin(p["scheduled"].keys())
            
    print("FILENAME:", filename)
    for priority, group in data.groupby("priority"):
        print("Priority:", priority)
        print("Num Requests:", len(group))
        print("Baseline:", group["baseline"].sum(), f"({round(group['baseline'].sum()/len(group)*100, 2)}%)")
        print("Scheduled:", group["scheduled"].sum(), f"({round(group['scheduled'].sum()/len(group)*100, 2)}%)")
        print("Perfect:", group["perfect"].sum(), f"({round(group['perfect'].sum()/len(group)*100, 2)}%)")
        print()
    print("===\n\n")

C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["priority"] = data["proposal_id"].apply(lambda x: proposals[x])
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

FILENAME: pb_2020-08_100-30-10.pkl
Priority: 0
Num Requests: 3307
Baseline: 2281 (68.97%)
Scheduled: 2339 (70.73%)
Perfect: 2139 (64.68%)

Priority: 10
Num Requests: 2518
Baseline: 2017 (80.1%)
Scheduled: 1980 (78.63%)
Perfect: 1917 (76.13%)

Priority: 30
Num Requests: 1080
Baseline: 715 (66.2%)
Scheduled: 683 (63.24%)
Perfect: 687 (63.61%)

Priority: 100
Num Requests: 2349
Baseline: 1978 (84.21%)
Scheduled: 2116 (90.08%)
Perfect: 2109 (89.78%)

===




C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["priority"] = data["proposal_id"].apply(lambda x: proposals[x])
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

FILENAME: pb_2020-08_100-55-10.pkl
Priority: 0
Num Requests: 3307
Baseline: 2281 (68.97%)
Scheduled: 2288 (69.19%)
Perfect: 2111 (63.83%)

Priority: 10
Num Requests: 2518
Baseline: 2017 (80.1%)
Scheduled: 1947 (77.32%)
Perfect: 1911 (75.89%)

Priority: 55
Num Requests: 1080
Baseline: 715 (66.2%)
Scheduled: 712 (65.93%)
Perfect: 709 (65.65%)

Priority: 100
Num Requests: 2349
Baseline: 1978 (84.21%)
Scheduled: 2095 (89.19%)
Perfect: 2086 (88.8%)

===




C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["priority"] = data["proposal_id"].apply(lambda x: proposals[x])
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

...

C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["priority"] = data["proposal_id"].apply(lambda x: proposals[x])
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

FILENAME: pb_2022-02_100-30-10.pkl
Priority: 0
Num Requests: 3234
Baseline: 1459 (45.11%)
Scheduled: 1600 (49.47%)
Perfect: 986 (30.49%)

Priority: 10
Num Requests: 1976
Baseline: 1482 (75.0%)
Scheduled: 1491 (75.46%)
Perfect: 769 (38.92%)

Priority: 30
Num Requests: 1450
Baseline: 905 (62.41%)
Scheduled: 859 (59.24%)
Perfect: 776 (53.52%)

Priority: 100
Num Requests: 5052
Baseline: 4158 (82.3%)
Scheduled: 4415 (87.39%)
Perfect: 4390 (86.9%)

===




C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["priority"] = data["proposal_id"].apply(lambda x: proposals[x])
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

FILENAME: pb_2022-02_100-55-10.pkl
Priority: 0
Num Requests: 3234
Baseline: 1459 (45.11%)
Scheduled: 1485 (45.92%)
Perfect: 940 (29.07%)

Priority: 10
Num Requests: 1976
Baseline: 1482 (75.0%)
Scheduled: 1440 (72.87%)
Perfect: 756 (38.26%)

Priority: 55
Num Requests: 1450
Baseline: 905 (62.41%)
Scheduled: 874 (60.28%)
Perfect: 788 (54.34%)

Priority: 100
Num Requests: 5052
Baseline: 4158 (82.3%)
Scheduled: 4315 (85.41%)
Perfect: 4314 (85.39%)

===


FILENAME: pb_2022-02_30-20-10.pkl
Priority: 0
Num Requests: 3234
Baseline: 1459 (45.11%)
Scheduled: 1537 (47.53%)
Perfect: 996 (30.8%)

Priority: 10
Num Requests: 1976
Baseline: 1482 (75.0%)
Scheduled: 1452 (73.48%)
Perfect: 734 (37.15%)

Priority: 20
Num Requests: 1450
Baseline: 905 (62.41%)
Scheduled: 861 (59.38%)
Perfect: 790 (54.48%)

Priority: 30
Num Requests: 5052
Baseline: 4158 (82.3%)
Scheduled: 4263 (84.38%)
Perfect: 4206 (83.25%)

===




C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["priority"] = data["proposal_id"].apply(lambda x: proposals[x])
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2628582479.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

In [61]:
example = list(filemap.keys())[0]
i = pickle.load(open(filemap[example]["input"], "rb"))
o = pickle.load(open(filemap[example]["output"], "rb"))
p = pickle.load(open(filemap[example]["perfect"], "rb"))
b = pickle.load(open(filemap[example]["baseline"], "rb"))

In [63]:
proposals = i["proposals"]
for calib_name in calib_proposals:
    proposals[calib_name] = 0
proposals

{'OGG_calib': 0,
 'MuSCAT Commissioning': 0,
 'CON2020B-009': 30,
 'CON2020B-001': 30,
 'auto_focus': 0,
 'TAU2020B-002': 100,
 'TAU2020B-008': 30,
 'TAU2020B-012': 10,
 'KEY2020B-006': 100,
 'LCOEngineering': 0,
 'NOAO2020B-015': 10,
 'LCO2020B-003': 100,
 'TAU2020B-004': 100,
 'SUPA2020B-006': 10,
 'LCO2020B-001': 100,
 'NOAO2020B-007': 30,
 'ANU2020B-001': 100,
 'NOAO2020B-011': 10,
 'FTP2020B-003': 10,
 'DDT2020B-010': 10,
 'CON2020B-003': 30,
 'FTPEPO2014A-004': 10,
 'COJ_calib': 0,
 'CON2020B-005': 30,
 'KEY2020B-007': 10,
 'HAW2020B-001': 100,
 'NOAO2020B-005': 100,
 'NOAO2020B-009': 10,
 'LCO2020B-006': 100,
 'LCOEPO2014B-010': 10,
 'NOAO2020B-012': 10,
 'HAW2020B-002': 10,
 'KEY2020B-003': 100,
 'KEY2020B-002': 100,
 'KEY2020B-005': 100}

In [64]:
data = i["all_requests"][["id", "proposal_id"]]
data["priority"] = data["proposal_id"].apply(lambda x: proposals[x])
data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
data["baseline"] = data["id"].isin(b["final_completed_requests"].keys())
data["perfect"] = data["id"].isin(p["scheduled"].keys())
data

C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2170045194.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["priority"] = data["proposal_id"].apply(lambda x: proposals[x])
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2170045194.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_852\2170045194.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a D

,id,proposal_id,priority,scheduled,baseline,perfect
2373081,2373081,NOAO2020B-011,10,False,False,False
2373019,2373019,KEY2020B-005,100,True,True,True
2373018,2373018,NOAO2020B-011,10,False,False,False
2373014,2373014,NOAO2020B-011,10,False,False,False
2372999,2372999,auto_focus,0,False,False,False
...,...,...,...,...,...,...
2165155,2165155,FTP2020B-003,10,True,True,True
2165156,2165156,FTP2020B-003,10,True,True,True
2165157,2165157,FTP2020B-003,10,False,False,False
2165158,2165158,FTP2020B-003,10,True,True,True


In [78]:
for priority, group in data.groupby("priority"):
    print("Priority:", priority)
    print("Num Requests:", len(group))
    print("Baseline:", group["baseline"].sum(), f"({round(group['baseline'].sum()/len(group)*100, 2)}%)")
    print("Scheduled:", group["scheduled"].sum(), f"({round(group['scheduled'].sum()/len(group)*100, 2)}%)")
    print("Perfect:", group["perfect"].sum(), f"({round(group['perfect'].sum()/len(group)*100, 2)}%)")
    print()

Priority: 0
Num Requests: 3307
Baseline: 2281 (68.97%)
Scheduled: 2339 (70.73%)
Perfect: 2139 (64.68%)

Priority: 10
Num Requests: 2518
Baseline: 2017 (80.1%)
Scheduled: 1980 (78.63%)
Perfect: 1917 (76.13%)

Priority: 30
Num Requests: 1080
Baseline: 715 (66.2%)
Scheduled: 683 (63.24%)
Perfect: 687 (63.61%)

Priority: 100
Num Requests: 2349
Baseline: 1978 (84.21%)
Scheduled: 2116 (90.08%)
Perfect: 2109 (89.78%)



In [72]:
counter = 0
for i, row in data.iterrows():
    if not np.all(row[["scheduled", "baseline"]]):
        if np.any(row[["scheduled", "baseline"]]):
            print(row[["id", "scheduled", "baseline"]].tolist())
            counter += 1
print(counter)

[2372934, True, False]
[2372883, False, True]
[2371841, True, False]
[2371756, True, False]
[2371625, False, True]
[2370487, True, False]
[2370456, True, False]
[2370404, True, False]
[2370401, True, False]
[2370349, False, True]
[2369305, False, True]
[2369170, True, False]
[2369169, True, False]
[2369165, True, False]
[2369149, True, False]
[2369086, True, False]
[2368802, True, False]
[2368798, False, True]
[2368585, False, True]
[2368474, False, True]
[2368261, True, False]
[2368039, True, False]
[2368006, True, False]
[2367906, False, True]
[2367904, False, True]
[2367821, False, True]
[2367566, True, False]
[2367461, False, True]
[2367140, False, True]
[2367088, False, True]
[2367030, False, True]
[2366908, True, False]
[2366418, True, False]
[2366259, True, False]
[2366253, True, False]
[2366252, True, False]
[2366249, False, True]
[2366235, False, True]
[2366144, False, True]
[2366143, False, True]
[2366141, False, True]
[2366124, True, False]
[2366117, False, True]
[2366026, T

* Get all requests [DONE]
* Get proposals [DONE]
* Filter out calibration proposals (see which ones were filtered out when constructing the input) [DONE]
* Assign group category to all remaining proposals [DONE]
* Filter out requests that belong to calibration proposals [DONE]
* Assign a success tag based on output data [DONE]
* Assign a perfect success tag based on the perfect data [DONE]
* plot?